# Kutsenko Solar Active-Region ML Benchmark

This notebook documents the catalogue benchmark used in the project.

**Important:** this is a retrospective aggregate analysis. `Phi_max` is not an operational early-forecast feature because it is known only after the region evolves. The next project phase will use time-resolved HMI/SHARP measurements with strict leakage prevention.

### References
- Kutsenko et al. 2019 — DOI: 10.1093/mnras/stz308
- Kutsenko et al. 2021 — DOI: 10.1093/mnras/staa3548
- Kutsenko, Abramenko & Plotnikov 2024 — DOI: 10.1088/1674-4527/ad2e4d


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import make_scorer, brier_score_loss
from xgboost import XGBClassifier, XGBRegressor
import shap

DATA=Path("../data")
RESULTS=Path("../results")
RESULTS.mkdir(exist_ok=True)
print("Ready.")


## 1. Load Kutsenko 2021 model-ready catalogue

For Kaggle, replace the path with:

`/kaggle/input/datasets/sj3267/kutsenko/KUTSENKO_2021_243_AR_model_ready.csv`


In [ ]:
path=DATA/"KUTSENKO_2021_243_AR_model_ready.csv"
if not path.exists():
    path=Path("/kaggle/input/datasets/sj3267/kutsenko/KUTSENKO_2021_243_AR_model_ready.csv")
flare=pd.read_csv(path)
print(flare.shape)
print(flare.columns.tolist())
display(flare.head())


## 2. Binary flare-productivity classification

The project compares the same nonlinear model family across feature sets. Metrics are ROC-AUC, PR-AUC and Brier score.


In [ ]:
flare["flare_index"]=pd.to_numeric(flare["flare_index"],errors="coerce")
flare["flare_productive"]=(flare["flare_index"]>=1).astype(int)

SETS={
"R_av only":["R_av_1e20_Mx_per_h"],
"Phi only":["Phi_max_1e21_Mx"],
"Phi + R_av":["Phi_max_1e21_Mx","R_av_1e20_Mx_per_h"],
"R_av + alpha":["R_av_1e20_Mx_per_h","alpha_sat","alpha_mean"],
"R_av + alpha uncertainty":["R_av_1e20_Mx_per_h","alpha_sat","alpha_mean","alpha_sat_std","alpha_mean_std"],
"All features":["Phi_max_1e21_Mx","R_av_1e20_Mx_per_h","alpha_sat","alpha_mean","alpha_sat_std","alpha_mean_std"]
}

def brier(estimator,X,y):
    return brier_score_loss(y,estimator.predict_proba(X)[:,1])

cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
rows=[]
for name,features in SETS.items():
    d=flare[features+["flare_productive"]].replace([np.inf,-np.inf],np.nan)
    X,y=d[features],d["flare_productive"]
    model=Pipeline([
        ("imputer",SimpleImputer(strategy="median")),
        ("model",XGBClassifier(n_estimators=400,max_depth=2,learning_rate=0.03,
                               min_child_weight=5,subsample=0.8,colsample_bytree=0.8,
                               reg_alpha=0.5,reg_lambda=2.0,eval_metric="logloss",
                               random_state=42,n_jobs=-1))
    ])
    s=cross_validate(model,X,y,cv=cv,scoring={
        "roc_auc":"roc_auc","pr_auc":"average_precision",
        "brier":make_scorer(brier,response_method="predict_proba")
    },n_jobs=-1)
    rows.append({"Model":name,"ROC_AUC":s["test_roc_auc"].mean(),
                 "PR_AUC":s["test_pr_auc"].mean(),"Brier":s["test_brier"].mean()})
classification=pd.DataFrame(rows).sort_values("ROC_AUC",ascending=False)
display(classification)


## 3. Continuous flare-productivity regression

Target:

`log10(flare_index)`

for positive flare-index values.

The central controlled comparison is the feature ablation.


In [ ]:
reg=flare[flare["flare_index"]>0].copy()
reg["log_flare_index"]=np.log10(reg["flare_index"])

REG_SETS={
"Phi_max only":["Phi_max_1e21_Mx"],
"R_av only":["R_av_1e20_Mx_per_h"],
"Phi_max + R_av":["Phi_max_1e21_Mx","R_av_1e20_Mx_per_h"],
"Phi + R_av + alpha":["Phi_max_1e21_Mx","R_av_1e20_Mx_per_h","alpha_sat","alpha_mean"],
"All features":["Phi_max_1e21_Mx","R_av_1e20_Mx_per_h","alpha_sat","alpha_mean","alpha_sat_std","alpha_mean_std"]
}

cv=KFold(n_splits=5,shuffle=True,random_state=42)
rows=[]
for name,features in REG_SETS.items():
    d=reg[features+["log_flare_index"]].replace([np.inf,-np.inf],np.nan).dropna()
    X,y=d[features],d["log_flare_index"]
    model=Pipeline([
        ("imputer",SimpleImputer(strategy="median")),
        ("model",XGBRegressor(n_estimators=500,max_depth=2,learning_rate=0.03,
                              min_child_weight=5,subsample=0.8,colsample_bytree=0.8,
                              reg_alpha=0.5,reg_lambda=2.0,
                              objective="reg:squarederror",random_state=42,n_jobs=-1))
    ])
    s=cross_validate(model,X,y,cv=cv,scoring={
        "R2":"r2","MAE":"neg_mean_absolute_error","RMSE":"neg_root_mean_squared_error"
    },n_jobs=-1)
    rows.append({"Model":name,"N_features":len(features),
                 "CV_R2":s["test_R2"].mean(),"CV_R2_std":s["test_R2"].std(),
                 "CV_MAE":-s["test_MAE"].mean(),"CV_RMSE":-s["test_RMSE"].mean()})
regression=pd.DataFrame(rows).sort_values("CV_R2",ascending=False)
display(regression)


## 4. Interpretation-only all-feature XGBoost

This fit is used for model interpretation, not as the source of the final predictive-performance claim.


In [ ]:
features=REG_SETS["All features"]
d=reg[features+["log_flare_index"]].replace([np.inf,-np.inf],np.nan).dropna()
X,y=d[features],d["log_flare_index"]

model=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("model",XGBRegressor(n_estimators=500,max_depth=2,learning_rate=0.03,
                          min_child_weight=5,subsample=0.8,colsample_bytree=0.8,
                          reg_alpha=0.5,reg_lambda=2.0,
                          objective="reg:squarederror",random_state=42,n_jobs=-1))
])
model.fit(X,y)
xgb=model.named_steps["model"]

importance=pd.DataFrame({"feature":features,"importance":xgb.feature_importances_}).sort_values("importance",ascending=False)
display(importance)


In [ ]:
X_imp=model.named_steps["imputer"].transform(X)
X_shap=pd.DataFrame(X_imp,columns=features)
explainer=shap.TreeExplainer(xgb)
shap_values=explainer.shap_values(X_shap)

shap_importance=pd.DataFrame({
    "feature":features,
    "mean_abs_SHAP":np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_SHAP",ascending=False)

display(shap_importance)
shap.summary_plot(shap_values,X_shap,feature_names=features)


In [ ]:
for feature in ["Phi_max_1e21_Mx","alpha_sat","alpha_mean"]:
    shap.dependence_plot(feature,shap_values,X_shap,interaction_index="auto")


## 5. Interpretation

The strongest aggregate predictive signal is maximum unsigned magnetic flux. The alpha parameters show nonlinear associations in the fitted all-feature model, but controlled cross-validation does not show that adding them improves continuous regression beyond the flux-only model.

This distinction—**model interpretation versus out-of-sample predictive improvement**—is a central methodological point of the project.

The next phase uses time-resolved SHARP quantities and will enforce AR/HARP-level grouping and strict cutoff-based feature construction.
